# Assignment 4 - 816039282

*MLOps & Model Deployment*

## Assignment 2 Repeated code - Dataset Loading and Models Training

In [2]:
# Data Ingestion & Cleaning

import os
import requests
from urllib.parse import urlparse
import polars as pl
import pandas as pd

os.makedirs('data/raw', exist_ok=True)
files_to_download = {
    "Yellow Taxi Trip Data": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
    "Taxi Zone Lookup Table": "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
}

for file_name_base, url in files_to_download.items():
    response = requests.get(url)
    parsed_url = urlparse(url)
    original_extension = os.path.splitext(parsed_url.path)[1]
    local_file_name = f'data/raw/{file_name_base}{original_extension}'

    with open(local_file_name, 'wb') as f:
        f.write(response.content)
    print(f"{file_name_base} downloaded successfully to {local_file_name}.")


taxi_file = 'data/raw/Yellow Taxi Trip Data.parquet'
df = pl.read_parquet(taxi_file)
print(f'Yellow taxi data starts with {len(df):,} rows')

# Drop rows with nulls in critical columns
critical_cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'fare_amount', 'tip_amount', 'payment_type']
before = len(df)
df = df.drop_nulls(subset=critical_cols)
after = len(df)
print(f'{after:,} rows remain after removing {before - after:,} null rows')

# Remove trips with invalid distance or fare
before = len(df)
df = df.filter((pl.col('trip_distance') > 0) &
               (pl.col('fare_amount') > 0) &
               (pl.col('fare_amount') <= 500))
after = len(df)
print(f'{after:,} rows remain after removing {before - after:,} invalid trips')

# Remove rows where dropoff is before pickup
before = len(df)
df = df.filter(pl.col('tpep_dropoff_datetime') >= pl.col('tpep_pickup_datetime'))
after = len(df)
print(f'{after:,} rows remain after removing {before - after:,} dropoff errors')

# Filter for credit card payments only
before = len(df)
df = df.filter(pl.col('payment_type') == 1)
after = len(df)
print(f'{after:,} rows remain after filtering for credit card payments (removed {before - after:,})')

rows_removed = len(pl.read_parquet(taxi_file)) - len(df)
print(f'{rows_removed:,} rows removed overall')


df = df.to_pandas()

zones = pd.read_csv('data/raw/Taxi Zone Lookup Table.csv')
df = df.merge(zones[['LocationID', 'Borough']], left_on='PULocationID', right_on='LocationID', how='left')

df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

print("Data ready for Part 1: Feature Engineering")
print(f"Final dataset shape: {df.shape}")


Yellow Taxi Trip Data downloaded successfully to data/raw/Yellow Taxi Trip Data.parquet.
Taxi Zone Lookup Table downloaded successfully to data/raw/Taxi Zone Lookup Table.csv.
Yellow taxi data starts with 2,964,624 rows
2,964,624 rows remain after removing 0 null rows
2,869,684 rows remain after removing 94,940 invalid trips
2,869,628 rows remain after removing 56 dropoff errors
2,298,380 rows remain after filtering for credit card payments (removed 571,248)
666,244 rows removed overall
Data ready for Part 1: Feature Engineering
Final dataset shape: (2298380, 21)


In [3]:
# Part 1.1: Feature Engineering
import numpy as np

# Temporal features
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['pickup_day_of_week'] = df['tpep_pickup_datetime'].dt.dayofweek  # 0=Monday
df['is_weekend'] = df['pickup_day_of_week'].isin([5, 6]).astype(int)

# Trip features
df['trip_duration_minutes'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60
df['trip_speed_mph'] = df['trip_distance'] / (df['trip_duration_minutes'] / 60).replace(0, np.nan)
df['log_trip_distance'] = np.log1p(df['trip_distance'])

# Fare features
df['fare_per_mile'] = df['fare_amount'] / df['trip_distance'].replace(0, np.nan)
df['fare_per_minute'] = df['fare_amount'] / df['trip_duration_minutes'].replace(0, np.nan)

# Zone features
zones = pd.read_csv('data/raw/Taxi Zone Lookup Table.csv')
borough_map = zones.set_index('LocationID')['Borough'].to_dict()

df['pickup_borough'] = df['PULocationID'].map(borough_map)
df['dropoff_borough'] = df['DOLocationID'].map(borough_map)

df = pd.get_dummies(df, columns=['pickup_borough', 'dropoff_borough'], prefix=['PU', 'DO'])

print("Feature engineering complete")
print("Feature columns created:", [col for col in df.columns if col not in ['tpep_pickup_datetime','tpep_dropoff_datetime']][:15], "...")


Feature engineering complete
Feature columns created: ['VendorID', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount'] ...


In [4]:
# Target variable for regression
df['tip_amount'] = df['tip_amount']

print("Target variable created")
print("Tip amount sample:", df['tip_amount'].head())


Target variable created
Tip amount sample: 0    3.75
1    3.00
2    2.00
3    3.20
4    6.90
Name: tip_amount, dtype: float64


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

numeric_features = [
    'pickup_hour', 'pickup_day_of_week', 'trip_duration_minutes',
    'trip_speed_mph', 'log_trip_distance', 'fare_per_mile', 'fare_per_minute']

X = df[numeric_features + [col for col in df.columns if col.startswith('PU_') or col.startswith('DO_')]]
y_regression = df['tip_amount']

# Split into train/val/test (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_regression, test_size=0.30, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42)

# Drop rows with NaNs
def drop_nan_rows(X, y):
    mask = ~X.isna().any(axis=1)
    return X[mask], y[mask]

X_train, y_train = drop_nan_rows(X_train, y_train)
X_val, y_val = drop_nan_rows(X_val, y_val)
X_test, y_test = drop_nan_rows(X_test, y_test)

# Scale numeric features
scaler = StandardScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_val[numeric_features] = scaler.transform(X_val[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

print("Data splitting and scaling complete")
print(f"Training set: {X_train.shape}, Validation set: {X_val.shape}, Test set: {X_test.shape}")


Data splitting and scaling complete
Training set: (1608841, 21), Validation set: (344753, 21), Test set: (344753, 21)


In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

sample_size = 200000
X_train_use = X_train.sample(sample_size, random_state=42)
y_train_use = y_train.loc[X_train_use.index]
print(f"Using {sample_size} rows for baseline regression training")

# Baseline regression models
lin_reg = LinearRegression().fit(X_train_use, y_train_use)
rf_reg = RandomForestRegressor(n_estimators=10, random_state=42, n_jobs=-1).fit(X_train_use, y_train_use)

# Evaluate regression models
for name, model in [('Linear Regression', lin_reg), ('Random Forest Regressor', rf_reg)]:
    y_pred = model.predict(X_val)
    print(f"\n{name} Validation Performance:")
    print("MAE:", mean_absolute_error(y_val, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_val, y_pred)))
    print("R²:", r2_score(y_val, y_pred))


Using 200000 rows for baseline regression training

Linear Regression Validation Performance:
MAE: 1.4229106632196984
RMSE: 2.6040039071441763
R²: 0.5364946297601825

Random Forest Regressor Validation Performance:
MAE: 1.3446240542331687
RMSE: 2.557234720362781
R²: 0.5529946799287013


In [7]:
from sklearn.model_selection import GridSearchCV

sample_size = 200000
X_train_tune = X_train.sample(sample_size, random_state=42)
y_train_tune = y_train.loc[X_train_tune.index]
print(f"Using {sample_size} rows for regression tuning")

# Regression tuning
param_grid_reg = {
    'n_estimators': [10, 20],
    'max_depth': [None, 5],
    'min_samples_split': [2, 5]
}
grid_reg = GridSearchCV(RandomForestRegressor(random_state=42),
                        param_grid_reg, cv=2, n_jobs=-1)
grid_reg.fit(X_train_tune, y_train_tune)
print("Best RF Regressor params:", grid_reg.best_params_)


Using 200000 rows for regression tuning
Best RF Regressor params: {'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 10}


In [8]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Evaluate regression models on test set
regression_results_test = {
    "Model": ["Linear Regression", "Random Forest Regressor", "Tuned RF Regressor"],
    "MAE": [
        mean_absolute_error(y_test, lin_reg.predict(X_test)),
        mean_absolute_error(y_test, rf_reg.predict(X_test)),
        mean_absolute_error(y_test, grid_reg.best_estimator_.predict(X_test)) if 'grid_reg' in globals() else None],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, lin_reg.predict(X_test))),
        np.sqrt(mean_squared_error(y_test, rf_reg.predict(X_test))),
        np.sqrt(mean_squared_error(y_test, grid_reg.best_estimator_.predict(X_test))) if 'grid_reg' in globals() else None],
    "R²": [
        r2_score(y_test, lin_reg.predict(X_test)),
        r2_score(y_test, rf_reg.predict(X_test)),
        r2_score(y_test, grid_reg.best_estimator_.predict(X_test)) if 'grid_reg' in globals() else None]
}
regression_df_test = pd.DataFrame(regression_results_test)
print("\nRegression Models - Test Set Comparison:")
print(regression_df_test)



Regression Models - Test Set Comparison:
                     Model       MAE      RMSE        R²
0        Linear Regression  1.428079  2.698751  0.523257
1  Random Forest Regressor  1.349994  2.643988  0.542409
2       Tuned RF Regressor  1.280121  2.544790  0.576101


In [9]:
# Random Forest feature importance
importances = rf_reg.feature_importances_
indices = np.argsort(importances)[::-1]

# Linear Regression coefficients
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": lin_reg.coef_
}).sort_values(by="Coefficient", ascending=False)
print("\nLinear Regression Coefficients:")
print(coef_df)



Linear Regression Coefficients:
                  Feature   Coefficient
16                 DO_EWR  3.685418e+00
4       log_trip_distance  2.221337e+00
19       DO_Staten Island  1.775543e+00
6         fare_per_minute  2.150540e-01
0             pickup_hour  8.665722e-02
5           fare_per_mile  8.496355e-02
2   trip_duration_minutes  1.758527e-02
12       PU_Staten Island -1.065814e-14
1      pickup_day_of_week -6.270749e-02
3          trip_speed_mph -8.247578e-02
11              PU_Queens -1.132948e+00
10           PU_Manhattan -3.454846e+00
13             PU_Unknown -3.571077e+00
18              DO_Queens -3.653176e+00
20             DO_Unknown -4.006924e+00
17           DO_Manhattan -4.193269e+00
15            DO_Brooklyn -4.515838e+00
14               DO_Bronx -6.092068e+00
8             PU_Brooklyn -7.435896e+00
7                PU_Bronx -9.488597e+00
9                  PU_EWR -9.752862e+00


## Part 1-Experiment Tracking with MLflow

### Part 1.1-MLflow Setup & Experiment Logging

*This step initializes MLflow by setting the local tracking directory and creating a new experiment for taxi‑tip prediction.*

In [ ]:
#Step 1
import mlflow
import mlflow.sklearn

# Start local MLflow tracking server(default is ./mlruns folder)
mlflow.set_tracking_uri("file:./mlruns")

# Create MLflow experiment
mlflow.set_experiment("taxi-tip-prediction")

<Experiment: artifact_location='file:c:/Users/shani/Downloads/COMP3610_Assignment-4/mlruns/877646091967902175', creation_time=1776152149003, experiment_id='877646091967902175', last_update_time=1776152149003, lifecycle_stage='active', name='taxi-tip-prediction', tags={'mlflow.experimentKind': 'custom_model_development'}>

*This step trains a Random Forest model on the taxi‑tip dataset, evaluates its performance, and logs parameters, metrics, and artifacts to MLflow for experiment tracking.*

In [11]:
# Step 2
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

with mlflow.start_run(run_name="RandomForestRegressor"):
    # Train model
    rf_reg = RandomForestRegressor(
        n_estimators=20,
        max_depth=5,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    )
    rf_reg.fit(X_train, y_train)

    # Evaluate
    y_pred = rf_reg.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # Log parameters, metrics, model, tags
    mlflow.log_params({
        "n_estimators": 20,
        "max_depth": 5,
        "min_samples_split": 2
    })
    mlflow.log_metrics({"MAE": mae, "RMSE": rmse, "R2": r2})
    mlflow.sklearn.log_model(rf_reg, "model")
    mlflow.set_tags({"model_type": "RandomForestRegressor", "dataset_version": "2024-01"})

print("Random Forest run logged to MLflow")


2026/04/16 02:38:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 02:39:08 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Random Forest run logged to MLflow


*This step trains a Linear Regression model on the taxi‑tip dataset, evaluates its performance, and logs parameters, metrics, and artifacts to MLflow for comparison with other runs.*

In [ ]:
#Step 3
from sklearn.linear_model import LinearRegression

with mlflow.start_run(run_name="LinearRegression"):
    # Train model
    lin_reg = LinearRegression()
    lin_reg.fit(X_train, y_train)

    # Evaluate
    y_pred = lin_reg.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # Log parameters, metrics, model, tags
    mlflow.log_params({"fit_intercept": lin_reg.fit_intercept})
    mlflow.log_metrics({"MAE": mae, "RMSE": rmse, "R2": r2})
    mlflow.sklearn.log_model(lin_reg, "model")
    mlflow.set_tags({"model_type": "LinearRegression", "dataset_version": "2024-01"})

print("Linear Regression run logged to MLflow")


2026/04/16 02:40:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 02:40:33 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Linear Regression run logged to MLflow


*This screenshot confirms that both model runs (Random Forest and Linear Regression) were successfully logged in MLflow.*

![MLflow UI screenshot](images/MLruns.png)

### Part 1.2-Model Comparison & Registry

*This screenshot from the MLflow UI shows the side‑by‑side comparison of the Random Forest and Linear Regression runs, including their parameters, metrics, and tags, which highlights differences in performance and configuration.*

![Comparison in MLflow UI screenshot1](images/Logs1.png)
![Comparison in MLflow UI screenshot1](images/Logs2.png)

*This step programmatically lists all runs in the taxi‑tip‑prediction experiment and displays their IDs, model types, and evaluation metrics side‑by‑side, serving as a verification of the screenshots and comparison results.*

In [13]:
# Step 4: List and compare runs- Verification for screenshots
import mlflow

# List all runs in the experiment
experiment = mlflow.get_experiment_by_name("taxi-tip-prediction")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Display runs side-by-side
runs[["run_id", "tags.model_type", "metrics.RMSE", "metrics.MAE", "metrics.R2"]]


,run_id,tags.model_type,metrics.RMSE,metrics.MAE,metrics.R2
0,07ad0232238640168520acf581814a7c,LinearRegression,2.697442,1.430342,0.523719
1,0bef8a92ed2f46c8be6e495f7cb03730,RandomForestRegressor,2.551818,1.284045,0.573756
2,0534fca82371449cbc0a15cae935c311,LinearRegression,2.697442,1.430342,0.523719
3,453987eddd254f26b641d9241d8f176f,RandomForestRegressor,2.551818,1.284045,0.573756


Based on the logged metrics, the RandomForestRegressor performed better than the Linear Regression model. It achieved lower error values (MAE = 1.28 vs. 1.43, RMSE = 2.55 vs. 2.69) and a higher R² score (0.57 vs. 0.52), meaning it explained more variance in the data and produced more accurate predictions overall. This indicates that the Random Forest model was the stronger choice for the taxi‑tip prediction task.

*This step selects the best model run based on lowest RMSE and registers it in the MLflow Model Registry under the name taxi‑tip‑regressor, making it available for versioning and deployment.*

In [14]:
#Step 5: Register best model in MLflow Model Registry
# Pick the best run (lowest RMSE)
best_run_id = runs.loc[runs['metrics.RMSE'].idxmin(), 'run_id']

model_uri = f"runs:/{best_run_id}/model"

# Register the model (no description here)
result = mlflow.register_model(
    model_uri=model_uri,
    name="taxi-tip-regressor"
)

print("Registered model:", result.name, "version:", result.version)



Registered model 'taxi-tip-regressor' already exists. Creating a new version of this model...
2026/04/16 03:18:36 WARNING mlflow.tracking._model_registry.fluent: Run with id 0bef8a92ed2f46c8be6e495f7cb03730 has no artifacts at artifact path 'model', registering model based on models:/m-4d673b0d3e3640258eafeddcded84ee3 instead


Registered model: taxi-tip-regressor version: 2


Created version '2' of model 'taxi-tip-regressor'.


*This step updates the registered model’s version in the MLflow Model Registry by adding a descriptive note that documents its performance metrics and identifies it as the Random Forest Regressor.*

In [15]:
from mlflow import client

client = mlflow.tracking.MlflowClient()

# Update the version description
client.update_model_version(
    name="taxi-tip-regressor",
    version=result.version,
    description="Random Forest Regressor with RMSE=2.55, MAE=1.28, R²=0.57"
)


<ModelVersion: aliases=[], creation_timestamp=1776323916548, current_stage='None', deployment_job_state=None, description='Random Forest Regressor with RMSE=2.55, MAE=1.28, R²=0.57', last_updated_timestamp=1776323948384, metrics=[<Metric: dataset_digest=None, dataset_name=None, key='MAE', model_id='m-4d673b0d3e3640258eafeddcded84ee3', run_id='0bef8a92ed2f46c8be6e495f7cb03730', step=0, timestamp=1776321534114, value=1.2840449482495577>,
 <Metric: dataset_digest=None, dataset_name=None, key='R2', model_id='m-4d673b0d3e3640258eafeddcded84ee3', run_id='0bef8a92ed2f46c8be6e495f7cb03730', step=0, timestamp=1776321534114, value=0.5737562567586936>,
 <Metric: dataset_digest=None, dataset_name=None, key='RMSE', model_id='m-4d673b0d3e3640258eafeddcded84ee3', run_id='0bef8a92ed2f46c8be6e495f7cb03730', step=0, timestamp=1776321534114, value=2.551818157647236>], model_id='m-4d673b0d3e3640258eafeddcded84ee3', name='taxi-tip-regressor', params={'max_depth': '5', 'min_samples_split': '2', 'n_estimator

This output confirms that the best model (Random Forest Regressor) was successfully registered in the MLflow Model Registry, showing its version, parameters, and logged metrics (RMSE, MAE, R²).

*This step loads the registered model from the MLflow Model Registry and uses a sample row from the test set to generate a prediction, verifying that the model can be retrieved and applied successfully.*

In [16]:
# Step 6: Load the registered model and make a sample prediction
from mlflow import pyfunc
import pandas as pd

# Load the registered model (version 1)
model = pyfunc.load_model("models:/taxi-tip-regressor/1")

# Use a row from your test set to guarantee schema match
# Replace X_test with the actual variable name you used for your test features
sample = X_test.iloc[[0]]   # take the first row

# Make a prediction
prediction = model.predict(sample)
print("Sample input:\n", sample)
print("\nSample prediction:", prediction)



Sample input:
          pickup_hour  pickup_day_of_week  trip_duration_minutes  \
1747886    -1.116293             0.06548               1.753636   

         trip_speed_mph  log_trip_distance  fare_per_mile  fare_per_minute  \
1747886        0.062057           2.729274       -0.05522        -0.030406   

         PU_Bronx  PU_Brooklyn  PU_EWR  ...  PU_Queens  PU_Staten Island  \
1747886     False        False   False  ...       True             False   

         PU_Unknown  DO_Bronx  DO_Brooklyn  DO_EWR  DO_Manhattan  DO_Queens  \
1747886       False     False        False   False          True      False   

         DO_Staten Island  DO_Unknown  
1747886             False       False  

[1 rows x 21 columns]

Sample prediction: [13.64042088]


This output shows the sample input row taken from the test set and the corresponding prediction generated by the registered Random Forest model, confirming that the model loads correctly from MLflow and produces a tip estimate (≈ 13.64).

## Part 2-Model Serving with FastAPI

### Part 2.1-API Design & Implementation

*This screenshot confirms that the /predict endpoint was created successfully. The auto‑generated documentation shows the required JSON request body with trip features, validating that the API design and input schema are correctly implemented.*

![Single Reuest](images/SingleRequest.png)

*This screenshot shows the documented response codes for the /predict endpoint. A 200 response indicates a successful prediction, while a 422 response demonstrates FastAPI’s validation error handling. This confirms that the API enforces input constraints and communicates errors clearly, meeting the input validation and error handling requirements of Task 2.1.*

![Response Types](images/ResponseTypes.png)

*This shows a POST request to /predict with sample trip features. The API returns a 200 response including a unique prediction_id, the model_version (2), and the predicted tip_amount (rounded to 2 decimals). This confirms the model loads once at startup and serves predictions correctly.*

![Successful Request](images/SuccessfulResponse.png)

### Part 2.2-Enhanced API Features

*This shows the /predict/batch endpoint in the FastAPI docs. The request body accepts an array of trip feature objects, confirming that the API supports batch predictions with proper JSON schema.*

![Batch Request](images/BatchRequest.png)

*This shows the documented response codes for the batch endpoint. A 200 indicates successful predictions, while a 422 demonstrates FastAPI’s validation error handling. This confirms that the API enforces input constraints and communicates errors clearly.*

![Batch Response Types](images/BatchResponseTypes.png)

*This screenshot demonstrates a successful POST /predict/batch call. The API returns a 200 response with the model_version, a list of predictions each containing a unique prediction_id, and the predicted tip_amount. This validates correct batch processing and response formatting.*

![Batch Response](images/BatchResponse.png)

*This shows the /health endpoint automatically documented by FastAPI. The schema confirms that no parameters are required and that the endpoint returns a JSON response with status information. This demonstrates that a proper health check endpoint was implemented.*

![Health Request](images/Health.png)

*This screenshot demonstrates a successful GET /health call. The API returns a 200 response with status: "ok", model_loaded: true, and the model_version. This confirms that the server is running, the model is loaded once at startup, and the health check endpoint works as intended.*

![Health Response](images/HealthResponse.png)

*This shows the /model/info endpoint documented in FastAPI. The schema confirms that no parameters are required and that the endpoint returns a JSON response with model metadata. This demonstrates that the API exposes model information for transparency and reproducibility.*

![Model Response Type](images/ModelResponseType.png)

*This screenshot demonstrates a successful GET /model/info call. The API returns a 200 response with details such as the model_name, model_version, the list of feature_names, and evaluation metrics. This confirms that the endpoint returns clear json about the registered model, supporting traceability and validation.*

![ModelResponse](images/ModelResponse.png)

### Part 2.3-API Testing

*This shows that the FastAPI Swagger UI is accessible at http://127.0.0.1:8000/docs. The endpoints /predict, /predict/batch, /health, and /model/info are clearly listed, along with schemas for input validation. This confirms that the API documentation is automatically generated and available for testing.*

![Swagger UI](images/Docs.png)

*This screenshot demonstrates a successful test run using pytest. All five tests passed, confirming that the API endpoints behave as expected and meet the assignment requirements. The single deprecation warning relates to the httpx library and does not affect functionality.*

![pytests](images/pytests.png)

## Part 3-Containerization with Docker

### Part 3.1-Dockerfile & Image Building

![Image](images/Image.png)

### Part 3.2-Docker Compose & Deployment Demo